특징 선택(Feature Selection)과 차원 축소(Dimensionality Reduction)는 모두 모델의 성능을 높이고 과적합을 방지하기 위해 변수의 개수를 줄이는 기법이지만, 기존 변수를 다루는 방식에서 근본적인 차이가 있습니다.


In [1]:
import os
import joblib
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
# 1. 데이터 로드 및 분할
cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. 필수 전처리: 표준화 스케일링
scaler = StandardScaler()
X_data_scaled = scaler.fit(X_train)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'원본 변수 개수: {X.shape[1]}개\n')

'''
==========================================
[방법 A] 특징 선택 (Feature Selection: SelectKBest)
==========================================
'''

selector = SelectKBest(score_func=f_classif, k=5)
X_train_fs = selector.fit_transform(X_train_scaled, y_train)
X_test_fs = selector.transform(X_test_scaled)
selected_features = X.columns[selector.get_support()]
print('--- [특징 선택] ---')
print(f'선택된 특징 (5개): {list(selected_features)}')

'''
# max_iter=5000 추가하여 수렴 경고 해결
'''
model_fs = LogisticRegression(
solver='saga', l1_ratio=0.0, max_iter=5000, random_state=42
)
model_fs.fit(X_train_fs, y_train)
y_pred_fs = model_fs.predict(X_test_fs)
print(f'정확도(Accuracy): {accuracy_score(y_test, y_pred_fs):.4f}')
print(classification_report(y_test, y_pred_fs))

'''
==========================================
[방법 B] 차원 축소 (Feature Extraction: PCA)
==========================================
'''
pca = PCA(n_components=5, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
print('--- [차원 축소] ---')
print(
'PCA 설명된 분산 비율 합계 (정보 보존량):'
f' {pca.explained_variance_ratio_.sum():.4f}'
)

'''
max_iter=5000 추가하여 수렴 경고 해결
'''
model_pca = LogisticRegression(
solver='saga', l1_ratio=0.0, max_iter=5000, random_state=42
)
model_pca.fit(X_train_pca, y_train)
y_pred_pca = model_pca.predict(X_test_pca)
print(f'정확도(Accuracy): {accuracy_score(y_test, y_pred_pca):.4f}')
print(classification_report(y_test, y_pred_pca))

'''
==========================================
폴더 확인 후 두 모델 저장
==========================================
'''
save_dir = 'model'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
    
path_fs = os.path.join(save_dir, 'cancer_feature_selection_model.pkl')
joblib.dump(model_fs, path_fs)
path_pca = os.path.join(save_dir, 'cancer_pca_model.pkl')
joblib.dump(model_pca, path_pca)

print('=' * 40)
print(f'특징 선택 모델 저장 완료: {path_fs}')
print(f'차원 축소 모델 저장 완료: {path_pca}')


원본 변수 개수: 30개

--- [특징 선택] ---
선택된 특징 (5개): ['mean perimeter', 'mean concave points', 'worst radius', 'worst perimeter', 'worst concave points']
정확도(Accuracy): 0.9386
              precision    recall  f1-score   support

           0       0.89      0.95      0.92        42
           1       0.97      0.93      0.95        72

    accuracy                           0.94       114
   macro avg       0.93      0.94      0.93       114
weighted avg       0.94      0.94      0.94       114

--- [차원 축소] ---
PCA 설명된 분산 비율 합계 (정보 보존량): 0.8514
정확도(Accuracy): 0.9561
              precision    recall  f1-score   support

           0       0.93      0.95      0.94        42
           1       0.97      0.96      0.97        72

    accuracy                           0.96       114
   macro avg       0.95      0.96      0.95       114
weighted avg       0.96      0.96      0.96       114

특징 선택 모델 저장 완료: model\cancer_feature_selection_model.pkl
차원 축소 모델 저장 완료: model\cancer_pca_model.pkl


`SelectKBest`의 `score_func` 속성에는 문제 유형(분류 또는 회귀)과 데이터의 성격(연속형 또는 범주형)에 따라 `sklearn.feature_selection` 모듈에서 제공하는 적절한 통계 함수를 지정합니다.

**1. 분류 문제 (Classification Task)**

| **함수명** | **입력 특성 (X)** | **타겟 (y)** | **핵심 특징 및 적합한 상황** |
| --- | --- | --- | --- |
| **`f_classif`** | 연속형 (수치) | 범주형 | **ANOVA F-검정**. 클래스 간 특성의 평균 차이를 분석해 선형적 연관성을 측정합니다. (기본값으로 가장 흔히 사용) |
| **`chi2`** | 범주형 / 빈도수 | 범주형 | **카이제곱 검정**. 특성과 타겟 간의 카이제곱 통계량을 측정합니다. 텍스트 분류(TF-IDF 등)나 원-핫 인코딩 데이터에 유용합니다. |
| **`mutual_info_classif`** | 연속형 또는 범주형 | 범주형 | **상호 정보량 (Mutual Information)**. 엔트로피 기반으로 특성과 타겟 사이의 복잡한 **비선형 관계**까지 포착합니다. |

**2. 회귀 문제 (Regression Task)**

| **함수명** | **입력 특성 (X)** | **타겟 (y)** | **핵심 특징 및 적합한 상황** |
| --- | --- | --- | --- |
| **`f_regression`** | 연속형 (수치) | 연속형 | **피어슨 상관계수 기반 F-검정**. 특성과 타겟 간의 **선형 상관관계**를 빠르게 측정합니다. |
| **`mutual_info_regression`** | 연속형 또는 범주형 | 연속형 | **상호 정보량 (Mutual Information)**. 연속형 타겟과의 복잡한 **비선형 관계**를 파악할 때 사용합니다. |

**상황별 선택 기준**

- **수치형 데이터를 이용한 단순 분류:** `f_classif`
- **카운트/빈도수 데이터나 텍스트 데이터 분류:** `chi2`
- **수치형 데이터를 이용한 연속값 예측(회귀):** `f_regression`
- **데이터 간 복잡한 비선형 패턴이 존재하는 경우:** `mutual_info_classif` 또는 `mutual_info_regression` *(단, F-검정 방식에 비해 연산 속도가 느립니다.)*

## 1. Iris 데이터셋(seaborn)
- 전처리: 중복제거, IQR 기준 클리핑(이상치), StandardScaler 스케일링, PCA차원 축소

In [3]:
import os
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 데이터 로드
df = sns.load_dataset("titanic")

# ==========================================
# 1. 정제 (Cleansing)
# ==========================================

# 1-1. 결측치 처리
df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])

# 1-2. 이상치 처리 (IQR 클리핑)
q1 = df["fare"].quantile(0.25)
q3 = df["fare"].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
lower_bound = q1 - 1.5 * iqr
df["fare"] = np.clip(df["fare"], lower_bound, upper_bound)

# 1-3. 중복 데이터 제거 및 인덱스 재정렬 (중요: 인덱스 불일치 방지)
df = df.drop_duplicates().reset_index(drop=True)


# ==========================================
# 2. 변환 (Transformation)
# ==========================================

# 2-1. 범주형 데이터 인코딩 (One-Hot Encoding)
encoder = OneHotEncoder(sparse_output=False, drop="first")
encoded_sex = encoder.fit_transform(df[["sex"]])
encoded_sex_df = pd.DataFrame(
    encoded_sex, columns=encoder.get_feature_names_out(["sex"])
)

# 2-2. 스케일링 및 정규화
scaler = StandardScaler()
df[["age_scaled", "fare_scaled"]] = scaler.fit_transform(df[["age", "fare"]])


# ==========================================
# 3. 특징 선택 및 생성 (Feature Engineering)
# ==========================================

# 파생 변수 생성
df["family_size"] = df["sibsp"] + df["parch"] + 1

# 전처리 완료된 특징 및 타겟 결합
X_features = pd.concat(
    [df[["age_scaled", "fare_scaled", "family_size"]], encoded_sex_df], axis=1
)
y = df["survived"]


# ==========================================
# 4. 데이터 축소 및 분할 (Reduction & Split)
# ==========================================

# Train / Test 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y, test_size=0.2, random_state=42
)

# 차원 축소 (PCA)
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print("전처리 완료 데이터 크기:", X_features.shape)
print("PCA 축소 후 Train 데이터 크기:", X_train_pca.shape)


# ==========================================
# 5. 모델 학습 & 평가 (LogisticRegression)
# ==========================================

# 최신 scikit-learn 기준 L2 규제 적용 (l1_ratio=0.0)
model = LogisticRegression(solver="saga", l1_ratio=0.0, random_state=42)
model.fit(X_train_pca, y_train)
y_pred = model.predict(X_test_pca)

print("\n--- Titanic Classification Report ---")
print(classification_report(y_test, y_pred))


# ==========================================
# 6. 폴더 확인 후 모델 저장
# ==========================================

save_dir = "model"
if not os.path.exists(save_dir):
  os.makedirs(save_dir)

model_path = os.path.join(save_dir, "titanic_logistic_model.pkl")
joblib.dump(model, model_path)
print(f"모델 저장 완료: {model_path}")

전처리 완료 데이터 크기: (778, 4)
PCA 축소 후 Train 데이터 크기: (622, 2)

--- Titanic Classification Report ---
              precision    recall  f1-score   support

           0       0.58      0.94      0.72        86
           1       0.71      0.17      0.28        70

    accuracy                           0.60       156
   macro avg       0.64      0.56      0.50       156
weighted avg       0.64      0.60      0.52       156

모델 저장 완료: model\titanic_logistic_model.pkl
